<a href="https://colab.research.google.com/github/tbcreator71/Google-Colab-Notebooks/blob/ComfyUI_Notebooks/LTX25_L4_24GB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Version: v1.2.11-L4

**Updated:** 2026-11-11 | **Profile:** L4 24GB, LTX 2.5 Distilled INT8 + Licon MSR V2 (AVref) + FLUX 3 Klein INT8 + FLUX.2 Dev (32B Q4_K_M GGUF + FP4 Mistral + SageAttention), shared Drive model cache, Cloudflare Named Tunnel (`comfy.saitgrup.com`)

---

# ComfyUI + LTX 2.5 | L4 24GB INT8

Independent L4 testing notebook. The A100 BF16 notebook and its existing saved results remain unchanged. Models are run exclusively from fast local Colab disk. Google Drive model archive is **shared**, so identical VAE and upscaler weights are reused, but L4-only INT8 transformer and text encoder are stored under their own filenames. ComfyUI setup archive is also reused.

Run cells Settings → Restore/Install → Download → Launch. Open `LTX-2.5_T2V_I2V_Single_Stage_Distilled_L4_INT8.json` first, set 512×768 or less, ~5 seconds, and turn off prompt enhancement while benchmarking. More intensive workflows may run out of memory or be slow on L4. Output backups use a separate Drive subfolder `L4_INT8`.

No inference from mounted Drive. Colab L4 selection must be made in Runtime settings; notebook metadata is not a guarantee of assigned GPU.


In [1]:
!nvidia-smi

Sat Nov 21 16:08:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#@title 1. Settings
from pathlib import Path
WORKSPACE = Path("/content/ComfyUI")  # Always use Colab's FAST local VM disk for all inference
USE_DRIVE_MODEL_ARCHIVE = False  #@param {type:"boolean"}
ARCHIVE_ROOT = "/content/drive/MyDrive/LTX25-Colab-Model-Archive"  #@param {type:"string"}
RESTORE_SETUP_ARCHIVE = True  #@param {type:"boolean"}
# Restore ComfyUI and custom nodes as one compressed archive, avoiding slow small-file Drive I/O.
REBUILD_SETUP_ARCHIVE = False  #@param {type:"boolean"}
# Set True once when you want to reinstall latest ComfyUI + nodes and refresh the setup archive.
SAVE_OUTPUTS_TO_DRIVE = True  #@param {type:"boolean"}
INSTALL_PROMPT_ENHANCER = True  #@param {type:"boolean"}
INSTALL_LICON_MSR_V2 = True  #@param {type:"boolean"}
INSTALL_FLUX3_KLEIN = True  #@param {type:"boolean"}
INSTALL_FLUX2_DEV = False  #@param {type:"boolean"}
INSTALL_FLUX2_DEV_FAST = True  #@param {type:"boolean"}
ENABLE_SAGE_ATTENTION = True  #@param {type:"boolean"}
GPU_PROFILE = "L4_INT8"
print("GPU profile:", GPU_PROFILE)
print("Model persistence:", USE_DRIVE_MODEL_ARCHIVE, "| Setup archive:", RESTORE_SETUP_ARCHIVE)
print("Local disk:", WORKSPACE, "| Drive archive:", ARCHIVE_ROOT)


GPU profile: L4_INT8
Model persistence: True | Setup archive: True
Local disk: /content/ComfyUI | Drive archive: /content/drive/MyDrive/LTX25-Colab-Model-Archive


In [3]:
#@title 2. Restore or install ComfyUI + LTX nodes (local SSD)
import os, sys, subprocess, shutil, tarfile, time
from pathlib import Path
def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, check=True)

ARCHIVE = None
if USE_DRIVE_MODEL_ARCHIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARCHIVE = Path(ARCHIVE_ROOT)
    ARCHIVE.mkdir(parents=True, exist_ok=True)

# v1 archive excluded EVERY directory named 'models', including ComfyUI's source-code
# directory comfy/ldm/models. It cannot start ComfyUI, so never restore it again.
SETUP_TAR = ARCHIVE / "comfyui-setup-v2.tar.gz" if ARCHIVE else None
CORE_MARKER = WORKSPACE / "comfy" / "ldm" / "models" / "autoencoder.py"
if REBUILD_SETUP_ARCHIVE:
    shutil.rmtree(WORKSPACE, ignore_errors=True)

if not (WORKSPACE / "main.py").exists() and RESTORE_SETUP_ARCHIVE and SETUP_TAR and SETUP_TAR.is_file() and not REBUILD_SETUP_ARCHIVE:
    print("Restoring small-file ComfyUI setup from one archive on Drive...", flush=True)
    from tqdm.auto import tqdm
    with tarfile.open(SETUP_TAR, "r:gz") as tar:
        # A progress indicator per extracted member; model weights are never in this archive.
        # The tar is created by this notebook. Never restore untrusted archives.
        members = tar.getmembers()
        with tqdm(total=len(members), unit="file", desc="Restoring ComfyUI setup", mininterval=0.8) as bar:
            for member in members:
                tar.extract(member, path="/content", filter="data")
                bar.update(1)
    print("Setup restored:", WORKSPACE)

def clone_or_pull(url, directory):
    directory = Path(directory)
    if (directory / ".git").exists():
        # Existing installations are reused to avoid breaking a working ComfyUI setup.
        return
    if not directory.exists():
        run("git", "clone", "--depth", "1", url, directory)

# Self-heal the incomplete older setup archive, without deleting local model weights.
# A fresh shallow clone only repairs the ComfyUI source tree. User models and outputs stay in place.
if (WORKSPACE / "main.py").exists() and not CORE_MARKER.is_file():
    print("Incomplete ComfyUI source detected (missing comfy/ldm/models). Repairing source only...", flush=True)
    import tempfile
    with tempfile.TemporaryDirectory(prefix="comfy-source-repair-", dir="/content") as scratch:
        fresh = Path(scratch) / "ComfyUI"
        run("git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", fresh)
        # Replace application source to keep its modules at one coherent revision,
        # but preserve all model weights, extensions, generated videos, user settings and inputs.
        protected = {"models", "custom_nodes", "user", "output", "input", "temp"}
        for entry in fresh.iterdir():
            if entry.name in protected or entry.name == ".git":
                continue
            destination = WORKSPACE / entry.name
            if entry.is_dir():
                shutil.copytree(entry, destination, dirs_exist_ok=True)
            elif entry.is_file():
                shutil.copy2(entry, destination)
    if not CORE_MARKER.is_file():
        raise RuntimeError("Core ComfyUI source repair failed: comfy/ldm/models/autoencoder.py is still missing")
    print("Core source repaired. Existing local models and outputs were preserved.", flush=True)

run("nvidia-smi")
free_gib = shutil.disk_usage("/content").free / 2**30
print("Local free space:", round(free_gib, 1), "GiB")
if free_gib < 90:
    print("WARNING: full BF16 model set may exceed local disk capacity. Check Colab's VM disk.")
# Colab images normally include ffmpeg. Avoid a slow apt repository refresh unless required.
if shutil.which("ffmpeg"):
    print("ffmpeg already installed: skipping apt-get update/install.", flush=True)
else:
    print("ffmpeg missing; installing it (apt may take time)...", flush=True)
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-y", "-qq", "ffmpeg")
run(sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "requests", "safetensors")
clone_or_pull("https://github.com/comfyanonymous/ComfyUI.git", WORKSPACE)
run(sys.executable, "-m", "pip", "install", "-q", "-r", WORKSPACE / "requirements.txt")
custom = WORKSPACE / "custom_nodes"
clone_or_pull("https://github.com/ltdrdata/ComfyUI-Manager.git", custom / "ComfyUI-Manager")
clone_or_pull("https://github.com/Lightricks/ComfyUI-LTXVideo.git", custom / "ComfyUI-LTXVideo")
if INSTALL_LICON_MSR_V2:
    clone_or_pull("https://github.com/liconstudio/ComfyUI-LTX2.5-MSR.git", custom / "ComfyUI-LTX2.5-MSR")
    clone_or_pull("https://github.com/kijai/ComfyUI-PromptRelay.git", custom / "ComfyUI-PromptRelay")
if INSTALL_FLUX2_DEV_FAST:
    clone_or_pull("https://github.com/city96/ComfyUI-GGUF.git", custom / "ComfyUI-GGUF")
    run(sys.executable, "-m", "pip", "install", "-q", "gguf")
if ENABLE_SAGE_ATTENTION:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sageattention"], check=True)
        print("SageAttention installed.")
    except Exception as exc:
        print("SageAttention wheel unavailable, falling back to native SDPA:", exc)
ltx_req = custom / "ComfyUI-LTXVideo" / "requirements.txt"
if ltx_req.exists():
    run(sys.executable, "-m", "pip", "install", "-q", "-r", ltx_req)
for folder in ("diffusion_models", "text_encoders", "vae", "latent_upscale_models", "loras"):
    (WORKSPACE / "models" / folder).mkdir(parents=True, exist_ok=True)
(WORKSPACE / "user" / "default" / "workflows").mkdir(parents=True, exist_ok=True)
run(sys.executable, "-c", "import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))")

if SETUP_TAR and (REBUILD_SETUP_ARCHIVE or not SETUP_TAR.exists()):
    if not CORE_MARKER.is_file():
        raise RuntimeError("Refusing to archive incomplete ComfyUI source: " + str(CORE_MARKER))
    print("Saving one small-file setup archive to Drive for future sessions...", flush=True)
    temp_tar = SETUP_TAR.with_suffix(".tar.gz.partial")
    with tarfile.open(temp_tar, "w:gz") as tar:
        # Save code + custom nodes only, NOT large model weights, outputs or generated media.
        for root, dirs, files in os.walk(WORKSPACE):
            here = Path(root)
            # Only exclude top-level model/output directories. Nested "models" can be Python source!
            relative = here.relative_to(WORKSPACE)
            blocked = {"models", "output", "input", "temp", "user"}
            if relative == Path("."):
                dirs[:] = [d for d in dirs if d not in blocked and d not in (".git", "__pycache__", ".cache")]
            else:
                dirs[:] = [d for d in dirs if d not in (".git", "__pycache__", ".cache")]
            for fn in files:
                if fn.endswith((".pyc", ".safetensors", ".ckpt", ".pt")): continue
                file = here / fn
                tar.add(file, arcname=file.relative_to(Path("/content")), recursive=False)
    os.replace(temp_tar, SETUP_TAR)
    print("Saved:", SETUP_TAR)
print("Ready. ComfyUI operates entirely from", WORKSPACE)


Mounted at /content/drive
Restoring small-file ComfyUI setup from one archive on Drive...


Restoring ComfyUI setup:   0%|          | 0/1545 [00:00<?, ?file/s]

Setup restored: /content/ComfyUI
+ nvidia-smi
Local free space: 235.4 GiB
ffmpeg already installed: skipping apt-get update/install.
+ /usr/bin/python3 -m pip install -q -U huggingface_hub requests safetensors
+ /usr/bin/python3 -m pip install -q -r /content/ComfyUI/requirements.txt
+ git clone --depth 1 https://github.com/liconstudio/ComfyUI-LTX2.5-MSR.git /content/ComfyUI/custom_nodes/ComfyUI-LTX2.5-MSR
+ git clone --depth 1 https://github.com/kijai/ComfyUI-PromptRelay.git /content/ComfyUI/custom_nodes/ComfyUI-PromptRelay
+ git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
+ /usr/bin/python3 -m pip install -q gguf
SageAttention installed.
+ /usr/bin/python3 -m pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-LTXVideo/requirements.txt
+ /usr/bin/python3 -c import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))
Ready. ComfyUI operates entirely from /content/ComfyUI


In [4]:
#@title 3. Download LTX 2.5 INT8 models (reuse common A100 cache)
import os, json, shutil, requests, time, concurrent.futures
from pathlib import Path
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

# Compatible with official 2.5 two-stage, single-stage and audio-to-video graphs.
# Files from the same official HF subfolders are placed directly under ComfyUI/models.
MODEL_FILES = [
 ("Lightricks/LTX-2.5","diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"),
 ("Lightricks/LTX-2.5","text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"),
 ("Lightricks/LTX-2.5","vae/ltx-2.5-video-vae-conv-bf16.safetensors"),
 ("Lightricks/LTX-2.5","vae/ltx-2.5-audio-vae-bf16.safetensors"),
 ("Lightricks/LTX-2.5","latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"),
]
if INSTALL_PROMPT_ENHANCER:
 MODEL_FILES.append(("Comfy-Org/gemma-4","text_encoders/gemma4_e2b_it_int8_convrot.safetensors"))
if INSTALL_LICON_MSR_V2:
 MODEL_FILES.append(("LiconStudio/LTX-2.5-Multiple-Subject-Reference", "LTX-2.5-Licon-MSR-V2.safetensors", "loras/LTX-2.5-Licon-MSR-V2.safetensors"))
if INSTALL_FLUX3_KLEIN:
 MODEL_FILES.extend([
  ("black-forest-labs/flux-3-image-klein-comfy", "diffusion_models/flux3-image-klein-int8-convrot.safetensors"),
  ("black-forest-labs/flux-3-image-klein-comfy", "text_encoders/qwen3vl_4b_int8_convrot.safetensors"),
  ("black-forest-labs/flux-3-image-klein-comfy", "vae/flux2-vae.safetensors"),
 ])
if INSTALL_FLUX2_DEV:
 MODEL_FILES.extend([
  ("Comfy-Org/flux2-dev", "split_files/diffusion_models/flux2_dev_fp8mixed.safetensors", "diffusion_models/flux2_dev_fp8mixed.safetensors"),
  ("Comfy-Org/flux2-dev", "split_files/text_encoders/mistral_3_small_flux2_fp8.safetensors", "text_encoders/mistral_3_small_flux2_fp8.safetensors"),
  ("Comfy-Org/flux2-dev", "split_files/loras/Flux2TurboComfyv2.safetensors", "loras/Flux2TurboComfyv2.safetensors"),
  ("Comfy-Org/flux2-dev", "split_files/vae/flux2-vae.safetensors", "vae/flux2-vae.safetensors"),
 ])
if INSTALL_FLUX2_DEV_FAST:
 MODEL_FILES.extend([
  ("city96/FLUX.2-dev-gguf", "flux2-dev-Q4_K_M.gguf", "diffusion_models/flux2-dev-Q4_K_M.gguf"),
  ("Comfy-Org/flux2-dev", "split_files/text_encoders/mistral_3_small_flux2_fp4_mixed.safetensors", "text_encoders/mistral_3_small_flux2_fp4_mixed.safetensors"),
  ("Comfy-Org/flux2-dev", "split_files/loras/Flux2TurboComfyv2.safetensors", "loras/Flux2TurboComfyv2.safetensors"),
  ("Comfy-Org/flux2-dev", "split_files/vae/flux2-vae.safetensors", "vae/flux2-vae.safetensors"),
 ])

# Persistent Drive archive -> copy big weights onto FAST local disk; never infer from Drive.
archive = ARCHIVE if USE_DRIVE_MODEL_ARCHIVE else None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None

# Check model authorization BEFORE any expensive multi-GB downloads.
from huggingface_hub import HfApi
if not HF_TOKEN:
    raise RuntimeError("Missing HF_TOKEN. In Colab: Secrets (key icon) → add HF_TOKEN → enable notebook access. First accept the license at https://huggingface.co/Lightricks/LTX-2.5")
try:
    HfApi().model_info("Lightricks/LTX-2.5", token=HF_TOKEN)
    # model_info may be accessible without weight access, so test metadata for the actual gated file.
    from huggingface_hub import get_hf_file_metadata, hf_hub_url
    get_hf_file_metadata(hf_hub_url("Lightricks/LTX-2.5", "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors"), token=HF_TOKEN)
except Exception as exc:
    raise RuntimeError("LTX 2.5 gated model access is not authorized. Accept its license at huggingface.co/Lightricks/LTX-2.5 and provide a Read token in Colab Secrets (HF_TOKEN).") from exc
print("Hugging Face model access verified.")

def copy_with_progress(src, dst, label):
    """Buffered copy with infrequent progress updates (16 MiB chunks; no extra files)."""
    from tqdm.auto import tqdm
    total = src.stat().st_size
    with open(src, "rb", buffering=16 * 1024 * 1024) as inp, open(dst, "wb", buffering=16 * 1024 * 1024) as out:
        with tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024,
                  desc=label, mininterval=1.0, miniters=4, leave=True) as progress:
            while True:
                chunk = inp.read(16 * 1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
                progress.update(len(chunk))

def file_ok(p, minimum=100_000_000):
    return p.is_file() and p.stat().st_size > minimum

# Parallelize only 2 large weight transfers. Drive throttling makes high concurrency unreliable.
def install_weight(entry):
    if len(entry) == 3:
        repo_id, hf_filename, target_rel = entry
        relpath = Path(target_rel)
    else:
        repo_id, rel = entry
        hf_filename = rel
        relpath = Path(rel if repo_id != "Comfy-Org/gemma-4" else "text_encoders/" + Path(rel).name)
    target = WORKSPACE / "models" / relpath
    stored = archive / relpath if archive else None
    target.parent.mkdir(parents=True, exist_ok=True)
    if file_ok(target):
        return ("Already local", target.name)
    if stored and file_ok(stored):
        print("Restoring from Drive:", target.name, flush=True)
        tmp = target.with_name(target.name + ".partial")
        try:
            copy_with_progress(stored, tmp, 'Restore: ' + target.name)
            if tmp.stat().st_size != stored.stat().st_size:
                raise IOError("Incomplete Drive restore: " + target.name)
            os.replace(tmp, target)
        finally:
            if tmp.exists(): tmp.unlink()
        return ("Restored", target.name)
    print("Downloading from Hugging Face:", repo_id, relpath, flush=True)
    try:
        dl_dir = WORKSPACE / "models" if hf_filename == relpath.as_posix() else target.parent
        dl_path = Path(hf_hub_download(repo_id=repo_id, filename=hf_filename, local_dir=dl_dir, token=HF_TOKEN))
        if dl_path.resolve() != target.resolve():
            os.replace(dl_path, target)
    except HfHubHTTPError as exc:
        if getattr(exc.response, "status_code", None) in (401, 403):
            raise RuntimeError(f"Accept the gated license on Hugging Face for {repo_id} (https://huggingface.co/{repo_id}); ensure HF_TOKEN is set in Colab Secrets.") from exc
        raise
    if not file_ok(target):
        raise RuntimeError("Downloaded model missing or unexpectedly small: " + str(target))
    if stored:
        stored.parent.mkdir(parents=True, exist_ok=True)
        if not file_ok(stored):
            print("Backing up to Drive:", target.name, flush=True)
            tmp = stored.with_name(stored.name + ".partial")
            try:
                copy_with_progress(target, tmp, 'Archive: ' + target.name)
                if tmp.stat().st_size != target.stat().st_size:
                    raise IOError("Incomplete Drive archive: " + target.name)
                os.replace(tmp, stored)
            finally:
                if tmp.exists(): tmp.unlink()
    return ("Downloaded and archived" if stored else "Downloaded", target.name)

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(install_weight, MODEL_FILES))
for state, filename in results: print(state, ":", filename)
OFFICIAL = "https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.5/"
WORKFLOWS = [
 "LTX-2.5_T2V_I2V_Single_Stage_Distilled.json",
 "LTX-2.5_T2V_I2V_Two_Stage_Distilled.json",
 "LTX-2.5_A2V_Two_Stage_Distilled.json",
]
replacements = {
 "ltx-2.5-22b-distilled-transformer-bf16.safetensors":"ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
 "gemma4-12b-with-proj-ltx-2.5-bf16.safetensors":"gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
 "gemma4_e2b_it_bf16.safetensors":"gemma4_e2b_it_int8_convrot.safetensors",
 "ltx-2.5-video-vae-bf16.safetensors":"ltx-2.5-video-vae-conv-bf16.safetensors",
}
wf_dir = WORKSPACE / "user" / "default" / "workflows"
wf_dir.mkdir(parents=True, exist_ok=True)
for filename in WORKFLOWS:
    response = requests.get(OFFICIAL + filename, timeout=120)
    response.raise_for_status()
    data = response.text
    for old_name,new_name in replacements.items():
        data = data.replace(old_name,new_name)  # Replaces model names in nested official subgraphs
    json.loads(data)
    out_name=filename.replace(".json","_L4_INT8.json")
    (wf_dir/out_name).write_text(data)
    print("L4 workflow ready:",out_name)
if INSTALL_LICON_MSR_V2:
    msr_wf_url = "https://raw.githubusercontent.com/liconstudio/ComfyUI-LTX2.5-MSR/main/LTX2.5-MSR-sample-workflow-V2.json"
    msr_resp = requests.get(msr_wf_url, timeout=120)
    msr_resp.raise_for_status()
    msr_data = msr_resp.text
    for old_name, new_name in replacements.items():
        msr_data = msr_data.replace(old_name, new_name)
    msr_data = msr_data.replace("ltx2.5\\\\LTX-2.5-Licon-MSR-V2.safetensors", "LTX-2.5-Licon-MSR-V2.safetensors")
    json.loads(msr_data)
    (wf_dir / "LTX2.5-MSR-sample-workflow-V2_L4_INT8.json").write_text(msr_data)
    print("L4 MSR V2 workflow ready: LTX2.5-MSR-sample-workflow-V2_L4_INT8.json")
if INSTALL_FLUX3_KLEIN:
    for fwf in ("flux3_klein_int8_mi2i.json", "flux3_klein_int8_t2i.json", "flux3_klein_int8_i2i.json"):
        try:
            fwf_path = hf_hub_download(repo_id="black-forest-labs/flux-3-image-klein-comfy", filename="workflows/" + fwf, token=HF_TOKEN)
            shutil.copy2(fwf_path, wf_dir / fwf)
            print("FLUX 3 Klein workflow ready:", fwf)
        except Exception as exc:
            print("Skipped FLUX 3 Klein workflow", fwf, "->", exc)
print("Start with LTX-2.5_T2V_I2V_Single_Stage_Distilled_L4_INT8.json")
print("Shared archive:",archive)
print("Turn off prompt enhancement initially; two-stage may need more than 24GB VRAM.")


Hugging Face model access verified.
Restoring from Drive: ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors
Restoring from Drive: gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors


Restore: ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors:   0%|          | 0.00/20.0G [00:00<…

Restore: gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors:   0%|          | 0.00/14.3G [00:00<?, ?B…

KeyboardInterrupt: 

In [ ]:
#@title 4. Start ComfyUI using Cloudflare Named Tunnel (comfy.saitgrup.com)
import os, re, time, subprocess, urllib.request
from pathlib import Path

PUBLIC_HOSTNAME = "https://comfy.saitgrup.com"

try:
    from google.colab import userdata
    tunnel_token = (userdata.get("CLOUDFLARE_TUNNEL_TOKEN") or "").strip()
except Exception:
    tunnel_token = (os.environ.get("CLOUDFLARE_TUNNEL_TOKEN") or "").strip()

if not tunnel_token:
    raise RuntimeError(
        "Missing CLOUDFLARE_TUNNEL_TOKEN. In Colab: Secrets (key icon) → add CLOUDFLARE_TUNNEL_TOKEN → enable notebook access."
    )

if subprocess.run(["bash", "-lc", "command -v cloudflared"], capture_output=True).returncode:
    deb = Path("/content/cloudflared.deb")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", deb)
    subprocess.run(["dpkg", "-i", str(deb)], check=True)

# Restart-safe within the SAME runtime, but do not run this cell in parallel.
for key in ("COMFY_PROCESS", "CLOUDFLARED_PROCESS"):
    process = globals().get(key)
    if process is not None and process.poll() is None:
        process.terminate()
        try: process.wait(timeout=8)
        except subprocess.TimeoutExpired: process.kill()
time.sleep(2)

log = open("/content/comfyui.log", "a", buffering=1)
comfy_cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--reserve-vram", "2", "--enable-cors-header", PUBLIC_HOSTNAME]
if globals().get("ENABLE_SAGE_ATTENTION", False):
    try:
        import sageattention  # noqa: F401
        comfy_cmd.append("--use-sage-attention")
        print("Attention backend: SageAttention (Ada Lovelace accelerated)")
    except Exception:
        print("Attention backend: native PyTorch SDPA")
COMFY_PROCESS = subprocess.Popen(comfy_cmd, cwd=str(WORKSPACE), stdout=log, stderr=subprocess.STDOUT)

comfy_ready = False
for _ in range(150):
    if COMFY_PROCESS.poll() is not None:
        raise RuntimeError("ComfyUI exited unexpectedly. Inspect /content/comfyui.log (see diagnostics cell).")
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2).read()
        comfy_ready = True
        break
    except Exception:
        time.sleep(2)
if not comfy_ready:
    raise TimeoutError("ComfyUI did not become ready on http://127.0.0.1:8188; inspect /content/comfyui.log")

cf_log_path = Path("/content/cloudflared.log")
log_start_offset = cf_log_path.stat().st_size if cf_log_path.exists() else 0
tunnel_log = open(cf_log_path, "a", buffering=1)
tunnel_env = os.environ.copy()
tunnel_env["TUNNEL_TOKEN"] = tunnel_token
CLOUDFLARED_PROCESS = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "run"],
    stdout=tunnel_log, stderr=subprocess.STDOUT, env=tunnel_env)

tunnel_connected = False
for _ in range(90):
    if CLOUDFLARED_PROCESS.poll() is not None:
        break
    if cf_log_path.exists():
        with open(cf_log_path, "r", errors="replace") as f:
            f.seek(log_start_offset)
            recent_log = f.read()
        if "Registered tunnel connection" in recent_log or re.search(r"Connection [0-9a-f-]+ registered", recent_log):
            tunnel_connected = True
            break
    time.sleep(2)

# Re-verify local ComfyUI is still running and healthy
if COMFY_PROCESS.poll() is not None:
    raise RuntimeError("ComfyUI process terminated after startup. Inspect /content/comfyui.log.")
if not tunnel_connected or CLOUDFLARED_PROCESS.poll() is not None:
    raise RuntimeError("Cloudflare Named Tunnel did not establish an active connection. Inspect /content/cloudflared.log.")

print("ComfyUI local server: READY")
print("Cloudflare Tunnel: CONNECTED")
print(f"Public hostname: {PUBLIC_HOSTNAME}")
print("GPU profile: NVIDIA L4 24GB")
print("Model profile: LTX 2.5 Distilled INT8")
print("\nUse Workflows → Open and select LTX-2.5_T2V_I2V_Single_Stage_Distilled_L4_INT8.json")
print("Keep Colab runtime connected while generating. Stop runtime to stop billing.")


In [ ]:
#@title 5. Diagnostics (run only when something fails)
from pathlib import Path
import shutil, subprocess
print("Free local disk GiB:",round(shutil.disk_usage("/content").free / 2**30,1))
subprocess.run(["nvidia-smi"])
for name in ("comfyui.log","cloudflared.log"):
    p=Path("/content")/name
    if p.exists():
        print("\n---",name,"(last 70 lines) ---")
        print("\n".join(p.read_text(errors="replace").splitlines()[-70:]))

In [ ]:
#@title 6. Optional: back up finished videos to Google Drive (not used during inference)
from pathlib import Path
import shutil
if not SAVE_OUTPUTS_TO_DRIVE:
    print("Output backup disabled. Videos are in", WORKSPACE / "output")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    src = WORKSPACE / "output"
    dst = Path("/content/drive/MyDrive/LTX25-Colab-Outputs/L4_INT8")
    dst.mkdir(parents=True,exist_ok=True)
    copied=0
    for f in src.rglob("*"):
        if f.is_file():
            target=dst / f.relative_to(src)
            target.parent.mkdir(parents=True,exist_ok=True)
            if not target.exists() or f.stat().st_size!=target.stat().st_size:
                shutil.copy2(f,target)
                copied+=1
    print("New/changed files backed up:", copied, "Destination:",dst)